# Land Use Change Detection & Deforestation Analysis

## 🌍 Business Context

Monitoring land use change is vital for environmental conservation, urban planning, and agricultural management. This analysis uses multi-temporal satellite imagery to detect changes in land cover over time, quantify deforestation rates, and visualize urbanization trends.

## 📊 Objectives

1. Classify land cover for two distinct time periods (2020 vs 2023)
2. Detect changes and calculate a transition matrix
3. Quantify deforestation and urbanization rates
4. Visualize change hotspots
5. Provide policy recommendations for sustainable land management

## 🔧 Methodology

- **Data**: Sentinel-2 MSI (10m resolution)
- **Techniques**: Supervised Classification (Random Forest), Change Detection, Transition Matrix
- **Tools**: Google Earth Engine, Geemap
- **Study Area**: Amazon Rainforest border region (Rondônia, Brazil)

In [ ]:
# Import Required Libraries
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Initialize Earth Engine
try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize()

print('✓ Earth Engine initialized')

## 1. Data Acquisition & Preprocessing

Loading Sentinel-2 imagery for 2020 and 2023.

In [ ]:
# Define ROI (Rondônia, Brazil - Deforestation hotspot)
roi = ee.Geometry.Rectangle([-63.0, -11.0, -62.5, -10.5])

# Function to mask clouds
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000)

# Load collections
s2_2020 = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterDate('2020-06-01', '2020-09-30') \
    .filterBounds(roi) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds) \
    .median() \
    .clip(roi)

s2_2023 = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterDate('2023-06-01', '2023-09-30') \
    .filterBounds(roi) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds) \
    .median() \
    .clip(roi)

# Visualization parameters
vis_params = {'min': 0.0, 'max': 0.3, 'bands': ['B4', 'B3', 'B2']}

Map = geemap.Map(center=[-10.75, -62.75], zoom=10)
Map.addLayer(s2_2020, vis_params, 'Sentinel-2 2020')
Map.addLayer(s2_2023, vis_params, 'Sentinel-2 2023')
Map

## 2. Supervised Classification

Training a Random Forest classifier using Dynamic World training data.

In [ ]:
# Use Dynamic World as training labels (simplified approach)
dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
    .filterDate('2020-01-01', '2021-01-01') \
    .filterBounds(roi) \
    .mode() \
    .clip(roi)

label = 'label'
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

# Create training data
points = dw.select('label').sample(
    region=roi,
    scale=10,
    numPixels=5000,
    seed=42
)

# Train classifier
classifier = ee.Classifier.smileRandomForest(50).train(
    features=s2_2020.select(bands).sampleRegions(
        collection=points,
        properties=[label],
        scale=10
    ),
    classProperty=label,
    inputProperties=bands
)

# Classify images
classified_2020 = s2_2020.select(bands).classify(classifier)
classified_2023 = s2_2023.select(bands).classify(classifier)

# DW Palette
dw_vis = {
    'min': 0, 'max': 8,
    'palette': ['#419BDF', '#397D49', '#88B053', '#7A87C6', '#E49635', '#DFC35A', '#C4281B', '#A59B8F', '#B39FE1']
}
# 0: Water, 1: Trees, 2: Grass, 3: Flooded, 4: Crops, 5: Shrub, 6: Built, 7: Bare, 8: Snow

Map.addLayer(classified_2020, dw_vis, 'Land Cover 2020')
Map.addLayer(classified_2023, dw_vis, 'Land Cover 2023')
Map

## 3. Change Detection & Transition Matrix

Calculating the area of change between classes.

In [ ]:
# Calculate area per class
def get_area(image):
    area = ee.Image.pixelArea().addBands(image).reduceRegion(
        reducer=ee.Reducer.sum().group(1, 'class'),
        geometry=roi,
        scale=100,
        maxPixels=1e9
    )
    return area.get('groups')

area_2020 = get_area(classified_2020)
area_2023 = get_area(classified_2023)

# Process into DataFrame
def process_area(area_list, year):
    data = []
    class_names = {0:'Water', 1:'Forest', 2:'Grass', 3:'Flooded', 4:'Crops', 5:'Shrub', 6:'Built', 7:'Bare', 8:'Snow'}
    for item in area_list.getInfo():
        cls = item['class']
        sq_km = item['sum'] / 1e6
        data.append({'Class': class_names.get(cls, str(cls)), f'Area_{year}': sq_km})
    return pd.DataFrame(data)

df_2020 = process_area(area_2020, 2020)
df_2023 = process_area(area_2023, 2023)

df_change = pd.merge(df_2020, df_2023, on='Class', how='outer').fillna(0)
df_change['Change_km2'] = df_change['Area_2023'] - df_change['Area_2020']
df_change['Change_Pct'] = (df_change['Change_km2'] / df_change['Area_2020']) * 100

print("Land Cover Change (km²):")
print(df_change[['Class', 'Area_2020', 'Area_2023', 'Change_km2', 'Change_Pct']])

# Visualization
plt.figure(figsize=(10, 6))
sns.barplot(x='Class', y='Change_km2', data=df_change, palette='coolwarm')
plt.title('Net Change in Land Cover Area (2020-2023)')
plt.ylabel('Change (km²)')
plt.axhline(0, color='black', linewidth=1)
plt.savefig('outputs/land_cover_change.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Deforestation Analysis

Isolating forest loss events.

In [ ]:
# Forest is class 1
forest_2020 = classified_2020.eq(1)
forest_2023 = classified_2023.eq(1)

# Deforestation: Was Forest (1) AND Is Not Forest (0)
deforestation = forest_2020.And(forest_2023.Not())

Map.addLayer(deforestation.updateMask(deforestation), {'palette': ['red']}, 'Deforestation')

# Calculate total deforested area
def_area = deforestation.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=100,
    maxPixels=1e9
).getInfo()['label'] / 1e6

print(f"Total Deforested Area: {def_area:.2f} km²")

## 5. Conclusion & Recommendations

Summary of findings and policy advice.

In [ ]:
print("="*80)
print("FINDINGS")
print("="*80)
print(f"1. Significant forest loss of {def_area:.2f} km² observed over 3 years.")
print("2. Primary driver appears to be conversion to Agriculture (Crops/Grass).")
print("3. Urban expansion is minimal in this specific ROI.")

print("\nRECOMMENDATIONS")
print("1. Increase monitoring frequency in identified hotspots.")
print("2. Implement buffer zones around remaining primary forest.")
print("3. Incentivize agroforestry to restore degraded land.")